# WX-R1 · Experiment 5 · Exp 5 shared encoders + one-hot band identity · fixed 60 epochs

**Corrected architectural variant** (methodology log entry 57, `docs/band_identity.md`). The original shared-encoder
models are invariant to permuting the five band graphs (no band identity). This variant appends a one-hot band
indicator to every band's node features (`model.band_identity: onehot`). Everything else is identical to the
original experiment, which stays as the baseline in its own folder. - **4 variant(s):** `plv__shared__mean__bandid`, `plv__shared__attention__bandid`, `ciplv__shared__mean__bandid`, `ciplv__shared__attention__bandid`
- **Settings:** xu_dev_cv, xu_holdout (cross-dataset only if the DESIGN.md §7 gate passes)
- **Protocol (from the config, unchanged):** 1 repeats × 5-fold subject-level outer CV; fixed hyperparameters, no hyperparameter search and no inner CV: one fit per outer training fold (protocol `gnn_fixed_hparams_v1`); 60 epochs.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** matched wMNE counterpart of the D3 LCMV / Xu suite (methodology log entry 55, `docs/wmne_xu.md`). All
outputs are under `<outputs>/D3_WMNE_XU/`, with its own code pin `D3_WMNE_XU/CODE_COMMIT.txt`. The LCMV suite
(`D3_LCMV_XU/`: stores, results, pins), `graphsD3_v4_*` and D1/D2 are only read, never written. Safe to run while
LCMV experiments run in another Colab session (separate folders and pin; one GPU per session).
**Results:** `<outputs>/D3_WMNE_XU/experiments_A_xu_tenfold/wx10_exp5_bandid/` and
`.../experiments_B_all_windows/wx_exp5_bandid/` (new folders). Use a **GPU runtime** (T4).

In [ ]:
# ===== D3 wMNE / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_WX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The wMNE / Xu suite has its own code pin, <outputs>/D3_WMNE_XU/CODE_COMMIT.txt (all WX notebooks run
# that commit); the LCMV suite pin D3_LCMV_XU/CODE_COMMIT.txt and all other pins are only reported.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'WX']
     + (['--update-code'] if UPDATE_WX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
WX = OUT + '/D3_WMNE_XU'
print('suite outputs:', WX, '| LCMV reference (read only):', LX)

## 0. Code pin of the corrected variants (R1)
The original experiments and their pins are not touched. This notebook runs the commit recorded in
`D3_WMNE_XU/R1_CODE_COMMIT.txt`. On its first run it adopts the other suite's R1 commit if that exists, so LCMV and wMNE use
identical code; otherwise it records the branch head. It never moves while R1 work of this suite exists. Leave
`UPDATE_R1_CODE = False`.

In [ ]:
UPDATE_R1_CODE = False
LX, WX = OUT + '/D3_LCMV_XU', OUT + '/D3_WMNE_XU'
import subprocess, sys
from pathlib import Path
from adgraph.colab import resolve_code_pin

head = subprocess.check_output(['git', '-C', '/content/repo', 'rev-parse', f'origin/{BRANCH}'], text=True).strip()
own_pin, other_pin = Path(WX) / 'R1_CODE_COMMIT.txt', Path(LX) / 'R1_CODE_COMMIT.txt'
sibling = other_pin.read_text().split()[0] if other_pin.exists() and other_pin.read_text().split() else None
target = head if UPDATE_R1_CODE else (sibling or head)
has_work = any(any(Path(WX).glob(f'experiments_*/*bandid*/**/*.done.json')) for _ in [0])
R1_COMMIT, _ = resolve_code_pin(own_pin, target, update=UPDATE_R1_CODE, has_work=has_work)
_run(['git', '-C', '/content/repo', 'checkout', '-q', R1_COMMIT])
if not Path('/content/repo/tests/test_xu_audit.py').exists():
    raise RuntimeError(f'pin {R1_COMMIT[:10]} lacks the band-identity code: set UPDATE_R1_CODE = True and run again')
for m in [m for m in sys.modules if m == 'adgraph' or m.startswith('adgraph.')]:
    del sys.modules[m]
from adgraph.colab import sh
print('R1 code commit:', R1_COMMIT, '| other suite R1 commit:', sibling or '(not yet set)')
if sibling and sibling != R1_COMMIT:
    print('WARNING: LCMV and wMNE R1 pins differ; the comparison would mix code versions')

## 1. Status

In [ ]:
CONFIGS = {'A': 'configs/experiments/wx/tenfold/wx10_exp5_bandid.yaml', 'B': 'configs/experiments/wx/wx_exp5_bandid.yaml'}
for mode, cfg in CONFIGS.items():
    print('==== mode', mode)
    sh(f"python scripts/experiment.py status --config {cfg}")

## 2. Run mode A, then mode B (resumable)

In [ ]:
ERRORS = {}
for mode, cfg in CONFIGS.items():
    try:
        sh(f"python scripts/experiment.py run --config {cfg}")
    except RuntimeError as exc:
        ERRORS[mode] = exc

## 3. Results (saved outputs only)

In [ ]:
import glob
from pathlib import Path
import pandas as pd
for mode, cfg in CONFIGS.items():
    sh(f"python scripts/experiment.py report --config {cfg}")
    root = 'experiments_A_xu_tenfold' if mode == 'A' else 'experiments_B_all_windows'
    reports = sorted(glob.glob(f'{WX}/{root}/{Path(cfg).stem}/_reports/report_*'))
    if reports:
        display(pd.read_csv(reports[-1] + '/results_summary.csv'))
if ERRORS:
    raise RuntimeError(f'units did not complete in mode(s) {sorted(ERRORS)}; completed units are saved. Run all again.')